In [23]:
# ==========================================
# Jupyter 智能防缓存魔法指令 (防 JVM 锁死)
%load_ext autoreload
%autoreload 2
# ==========================================

import sys
import math
import numpy as np

# 1. 导入四大核心组件
sys.path.append('../src') 
from orekit_generator import OrekitOrbitGenerator
from geodesy_engine import WGS84GeodesyEngine
from attitude_control import DynamicAttitudeController
from sensor_optics import LimbOpticsSimulator

# 2. 导入底层 Java 库
from org.hipparchus.geometry.euclidean.threed import Vector3D # type: ignore
from org.orekit.bodies import GeodeticPoint # type: ignore

print("================ 临边大气探测：WGS84 高保真全链路系统启动 ================\n")

# 3. 初始化各大子系统
orbit_sys = OrekitOrbitGenerator(
    prop_model='HPOP', a=(6378137.0 + 600000.0), e=0.001, i=math.radians(97.79), 
    raan=0.0, arg_pe=math.radians(90), m0=0.0, gravity_degree=6, gravity_order=6
)
geodesy_sys = WGS84GeodesyEngine()
attitude_sys = DynamicAttitudeController(mount_pitch_deg=68.0, drift_rate_arcsec_s=0.05, jitter_3sigma_arcsec=1.5)
optics_sys = LimbOpticsSimulator(focal_length_mm=850.0, sensor_size_mm=32.5)

# 4. 生成基准星历
print("⏳ 正在推演基准轨道数据...")
df_ephem = orbit_sys.generate_ephemeris_dataframe(duration_sec=180, step_sec=10)
print("🚀 飞行推演开始！\n")

# 设定目标切点高度 (你可以修改回 105000.0)
TARGET_ALT_M = 1050000.0  

# 5. 打印专业遥测表头 (新增 Noise 列)
print(f"{'T(s)':<4} | {'Sat Position (Lat, Lon, Alt)':<30} | {'Tgt Point (Lat, Lon, Alt)':<30} | {'Illum(S|T)':<10} | {'Noise(°)':<8} | {'FOV Range (km)':<15}")
print("-" * 118)

# 6. 主循环推演
for index, row in df_ephem.iterrows():
    t = row['time_sec']
    date = orbit_sys.initial_date.shiftedBy(float(t))
    x, y, z = row['x'], row['y'], row['z']
    vx, vy, vz = row['vx'], row['vy'], row['vz']
    
    # [卫星本体状态]
    sat_lat, sat_lon, sat_alt_m = geodesy_sys.get_sat_lla(x, y, z, date)
    sat_eclipse = geodesy_sys.is_in_eclipse(x, y, z, date)
    
    # [姿态与补偿] - 调用包含物理扰动开关的新接口
    # 返回值完美解包为 3 个变量
    mirror_offset, absolute_los, noise_deg = attitude_sys.get_pointing_command(
        x, y, z, TARGET_ALT_M, geodesy_sys, date, 
        enable_noise=False,   # <--- 【核心开关】改为 True 即可注入热漂移与高频微振动！
        t_sec=t, 
        is_thrusting=False
    )
    
    # [光学切点高度范围]
    alt_min_m, alt_max_m = optics_sys.calculate_altitude_range(
        x, y, z, vx, vy, vz, absolute_los, geodesy_sys, date
    )
    
    # [靶心切点大地坐标解算]
    pos_ecef = Vector3D(float(x), float(y), float(z))
    vel_ecef = Vector3D(float(vx), float(vy), float(vz))
    x_dir, _, z_dir = optics_sys._build_local_orbital_frame(pos_ecef, vel_ecef)
    los_center = optics_sys._get_los_vector(x_dir, z_dir, absolute_los)
    
    tgt_lat, tgt_lon, tgt_alt_m = geodesy_sys.get_limb_tangent_lla(
        x, y, z, los_center.getX(), los_center.getY(), los_center.getZ(), date
    )
    
    # [切点光照状态] (利用 GeodeticPoint 转回 ECEF 严密探测)
    tgt_gp = GeodeticPoint(math.radians(tgt_lat), math.radians(tgt_lon), float(tgt_alt_m))
    tgt_ecef = geodesy_sys.earth.transform(tgt_gp)
    tgt_eclipse = geodesy_sys.is_in_eclipse(tgt_ecef.getX(), tgt_ecef.getY(), tgt_ecef.getZ(), date)
    
    # [格式化输出]
    str_sat = f"{sat_lat:+06.2f}, {sat_lon:+07.2f}, {sat_alt_m/1000.0:6.2f}"
    str_tgt = f"{tgt_lat:+06.2f}, {tgt_lon:+07.2f}, {tgt_alt_m/1000.0:6.2f}"
    str_illum = f"{'ECLP' if sat_eclipse else 'SUN '} | {'ECLP' if tgt_eclipse else 'SUN '}"
    
    print(f"{t:03.0f}s | {str_sat:<30} | {str_tgt:<30} | {str_illum:<10} | {noise_deg:+06.3f} | [{alt_max_m/1000.0:6.2f}~{alt_min_m/1000.0:6.2f}] -> 靶心: {tgt_alt_m/1000.0:.2f}km")

print("\n🎉 [SYSTEM] 全链路 WGS84 闭环验证成功！")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
================ 临边大气探测：WGS84 高保真全链路系统启动 ================

⏳ 正在推演基准轨道数据...
🚀 [Orekit 全能引擎] 正在使用 HPOP 模型推演轨道...
🚀 飞行推演开始！

T(s) | Sat Position (Lat, Lon, Alt)   | Tgt Point (Lat, Lon, Alt)      | Illum(S|T) | Noise(°) | FOV Range (km) 
----------------------------------------------------------------------------------------------------------------------
000s | +82.25, -127.21, 614.02        | +82.25, -127.21, 614.02        | SUN  | SUN  | +0.000 | [612.74~   nan] -> 靶心: 614.02km
010s | +82.22, -131.82, 614.01        | +82.22, -131.82, 614.01        | SUN  | SUN  | +0.000 | [612.74~   nan] -> 靶心: 614.01km
020s | +82.13, -136.36, 614.01        | +82.13, -136.36, 614.01        | SUN  | SUN  | +0.000 | [612.75~   nan] -> 靶心: 614.01km
030s | +82.00, -140.78, 614.00        | +82.00, -140.78, 614.00        | SUN  | SUN  | +0.000 | [612.75~   nan] -> 靶心: 614.00km
040s | +81.83, -145.04, 613.99        | +81.83

In [24]:
# ==========================================
# Jupyter 智能防缓存魔法指令 (防 JVM 锁死)
%load_ext autoreload
%autoreload 2
# ==========================================

import sys
import math
import numpy as np

# 1. 导入四大核心组件
sys.path.append('../src') 
from orekit_generator import OrekitOrbitGenerator
from geodesy_engine import WGS84GeodesyEngine
from attitude_control import DynamicAttitudeController
from sensor_optics import LimbOpticsSimulator

# 2. 导入底层 Java 库
from org.hipparchus.geometry.euclidean.threed import Vector3D # type: ignore
from org.orekit.bodies import GeodeticPoint # type: ignore

print("================ 临边大气探测：WGS84 高保真全链路系统启动 ================\n")

# 3. 初始化各大子系统
orbit_sys = OrekitOrbitGenerator(
    prop_model='HPOP', a=(6378137.0 + 600000.0), e=0.001, i=math.radians(97.79), 
    raan=0.0, arg_pe=math.radians(90), m0=0.0, gravity_degree=6, gravity_order=6
)
geodesy_sys = WGS84GeodesyEngine()
attitude_sys = DynamicAttitudeController(mount_pitch_deg=68.0, drift_rate_arcsec_s=0.05, jitter_3sigma_arcsec=1.5)
optics_sys = LimbOpticsSimulator(focal_length_mm=850.0, sensor_size_mm=32.5)

# 4. 生成基准星历
print("⏳ 正在推演基准轨道数据...")
df_ephem = orbit_sys.generate_ephemeris_dataframe(duration_sec=180, step_sec=10)
print("🚀 飞行推演开始！\n")

TARGET_ALT_M = 400000.0  

# ==========================================
# [模式切换] 提取 T=0 时刻的完美视角并锁死
# ==========================================
print("⚙️ [特殊工况] 载荷光机系统降级，进入【视轴死锁模式】...")
row0 = df_ephem.iloc[0]
date0 = orbit_sys.initial_date.shiftedBy(float(row0['time_sec']))
pos0 = Vector3D(float(row0['x']), float(row0['y']), float(row0['z']))

# 仅在第一秒算一次理想角度
LOCKED_LOS_DEG = attitude_sys.calc_ideal_nadir_angle(pos0, TARGET_ALT_M, geodesy_sys, date0)
print(f"🔒 镜子已锁死！全轨道强制固定下俯角: {LOCKED_LOS_DEG:.4f}°\n")


# 5. 打印专业遥测表头
print(f"{'T(s)':<4} | {'Sat Position (Lat, Lon, Alt)':<30} | {'Tgt Point (Lat, Lon, Alt)':<30} | {'Illum(S|T)':<10} | {'Noise(°)':<8} | {'FOV Range (km)':<15}")
print("-" * 118)

# 6. 主循环推演
for index, row in df_ephem.iterrows():
    t = row['time_sec']
    date = orbit_sys.initial_date.shiftedBy(float(t))
    x, y, z = row['x'], row['y'], row['z']
    vx, vy, vz = row['vx'], row['vy'], row['vz']
    
    sat_lat, sat_lon, sat_alt_m = geodesy_sys.get_sat_lla(x, y, z, date)
    sat_eclipse = geodesy_sys.is_in_eclipse(x, y, z, date)
    
    # ==========================================
    # [关键调用] 调用锁死接口，传入常量 LOCKED_LOS_DEG
    # ==========================================
    mirror_offset, absolute_los, noise_deg = attitude_sys.get_locked_pointing_command(
        LOCKED_LOS_DEG,
        enable_noise=False,   # <--- 同样可以随时改为 True 观察抖动叠加漂移的双重打击
        t_sec=t, 
        is_thrusting=False
    )
    
    alt_min_m, alt_max_m = optics_sys.calculate_altitude_range(
        x, y, z, vx, vy, vz, absolute_los, geodesy_sys, date
    )
    
    pos_ecef = Vector3D(float(x), float(y), float(z))
    vel_ecef = Vector3D(float(vx), float(vy), float(vz))
    x_dir, _, z_dir = optics_sys._build_local_orbital_frame(pos_ecef, vel_ecef)
    los_center = optics_sys._get_los_vector(x_dir, z_dir, absolute_los)
    
    tgt_lat, tgt_lon, tgt_alt_m = geodesy_sys.get_limb_tangent_lla(
        x, y, z, los_center.getX(), los_center.getY(), los_center.getZ(), date
    )
    
    tgt_gp = GeodeticPoint(math.radians(tgt_lat), math.radians(tgt_lon), float(tgt_alt_m))
    tgt_ecef = geodesy_sys.earth.transform(tgt_gp)
    tgt_eclipse = geodesy_sys.is_in_eclipse(tgt_ecef.getX(), tgt_ecef.getY(), tgt_ecef.getZ(), date)
    
    str_sat = f"{sat_lat:+06.2f}, {sat_lon:+07.2f}, {sat_alt_m/1000.0:6.2f}"
    str_tgt = f"{tgt_lat:+06.2f}, {tgt_lon:+07.2f}, {tgt_alt_m/1000.0:6.2f}"
    str_illum = f"{'ECLP' if sat_eclipse else 'SUN '} | {'ECLP' if tgt_eclipse else 'SUN '}"
    
    # 注意最后一列，靶心高度将直接暴露地球扁率带来的剧烈漂移
    print(f"{t:03.0f}s | {str_sat:<30} | {str_tgt:<30} | {str_illum:<10} | {noise_deg:+06.3f} | [{alt_max_m/1000.0:6.2f}~{alt_min_m/1000.0:6.2f}] -> 靶心: {tgt_alt_m/1000.0:.2f}km")

print("\n🎉 [SYSTEM] 全链路锁死工况推演完成！")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
================ 临边大气探测：WGS84 高保真全链路系统启动 ================

⏳ 正在推演基准轨道数据...
🚀 [Orekit 全能引擎] 正在使用 HPOP 模型推演轨道...
🚀 飞行推演开始！

⚙️ [特殊工况] 载荷光机系统降级，进入【视轴死锁模式】...
🔒 镜子已锁死！全轨道强制固定下俯角: 75.7622°

T(s) | Sat Position (Lat, Lon, Alt)   | Tgt Point (Lat, Lon, Alt)      | Illum(S|T) | Noise(°) | FOV Range (km) 
----------------------------------------------------------------------------------------------------------------------
000s | +82.25, -127.21, 614.02        | +74.03, -064.50, 398.65        | SUN  | SUN  | +0.000 | [364.44~430.36] -> 靶心: 398.65km
010s | +82.22, -131.82, 614.01        | +74.56, -065.71, 398.75        | SUN  | SUN  | +0.000 | [364.56~430.46] -> 靶心: 398.75km
020s | +82.13, -136.36, 614.01        | +75.09, -067.01, 398.85        | SUN  | SUN  | +0.000 | [364.67~430.56] -> 靶心: 398.85km
030s | +82.00, -140.78, 614.00        | +75.61, -068.39, 398.96        | SUN  | SUN  | +0.000 | [364.78~430.65]